# tiny-voca in Spanish: the bilingual voice (tiny-speech)

One small voice (tiny-voca, about 8 MB in the browser) that speaks Mexican Spanish and English, for MapSMA's answers. Code and
details: `tiny-speech/training/` in github.com/larsboesen-prog/ranked-feed (branch `claude/tiny-speech-bilingual-v7lyp0`):
`RECIPE.md`, `gpu/README.md`.

**MODE** (next cell): `"design"` (GPU, about 20 minutes): a few Mexican voices designed from descriptions read the site's
sentences in both languages, for the owner to pick one (PICK). `"teach"` (GPU T4 ×2, design output as input): the picked voice
reads about 7,600 training sentences, checked by Whisper, then features and alignment. `"learn"` (GPU, teach output as input):
training, the gates, the result for the page.
Kaggle: Accelerator **GPU T4**, Internet **on**, then **Save Version → Save & Run All**.

In [ ]:
MODE = "design"   # "design": candidate voices; "teach": the teacher reads the training sentences (…annotate); "learn": train, gates
PICK = "M1"       # the owner's voice from the design run (teach mode)
import os, sys, shutil, pathlib, json, subprocess, socket


def run(*cmd, env=None):
    """a command, its output shown as it comes; the notebook stops with an error if it fails"""
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"failed ({p.returncode}): {' '.join(map(str, cmd))}")


try:
    socket.getaddrinfo("github.com", 443)
    socket.getaddrinfo("huggingface.co", 443)
except OSError:
    raise RuntimeError("No internet in this session. Kaggle: Settings → Internet on (the account must be phone-verified).")
KAGGLE = os.path.exists("/kaggle/working")
OUT = pathlib.Path("/kaggle/working/tiny-voice") if KAGGLE else pathlib.Path("tiny-voice").resolve()
BASE = pathlib.Path("/kaggle/tmp") if KAGGLE else pathlib.Path(".").resolve()
BASE.mkdir(parents=True, exist_ok=True)
REPO = BASE / "ranked-feed"
OUT.mkdir(parents=True, exist_ok=True)
print({"mode": MODE, "kaggle": KAGGLE, "out": str(OUT), "free_gb": round(shutil.disk_usage(BASE).free / 1e9, 1)})
run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv")


In [ ]:
# the code (public repository) and the packages: Qwen3-TTS without touching this machine's torch, Whisper for the check
if not REPO.exists():
    run("git", "clone", "-q", "--depth", "1", "--branch", "claude/tiny-speech-bilingual-v7lyp0", "https://github.com/larsboesen-prog/ranked-feed.git", REPO)
version = subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %s"], capture_output=True, text=True, check=True).stdout
(OUT / "code-version.txt").write_text(version)
print(version)
GPU = REPO / "tiny-speech/training/gpu"
run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "qwen-tts==0.1.1")
run(sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.3", "accelerate==1.12.0", "librosa", "soundfile", "sox", "onnxruntime",
    "einops", "faster-whisper", "jiwer", "num2words")


In [ ]:
# this MODE's work. teach and learn are one training run in two Kaggle sessions (12 h each at most): teach leaves everything it
# made in its output (tsw.tar); learn takes that output as its input, unpacks it to the same place and carries on.
WORKD = BASE / "tsw"
if MODE == "design":
    run(sys.executable, GPU / "design.py", "--out", OUT / "design")
elif MODE in ("teach", "learn"):
    import torch
    env = {**os.environ, "WORK": str(WORKD), "PY": sys.executable, "GPUS": str(max(1, torch.cuda.device_count())), "HOURS": "11",
           "LIKE": "F2" if PICK.startswith("F") else "M2"}  # the released voice the new one starts from: the nearer, the better
    if MODE == "teach":
        # the picked voice's reference, kept in the repository (training/voice/<PICK>/es.*), for both languages: one voice
        kept = REPO / "tiny-speech/training/voice" / PICK
        assert (kept / "es.wav").exists(), f"teach: no reference for {PICK} in training/voice/"
        refs = BASE / "references" / PICK
        refs.mkdir(parents=True, exist_ok=True)
        for lang in ("es", "en"):
            shutil.copy(kept / "es.wav", refs / f"{lang}.wav")
            shutil.copy(kept / "es.txt", refs / f"{lang}.txt")
        env["REFS"] = str(refs)
        (OUT / "pick.json").write_text(json.dumps({"pick": PICK, "references": env["REFS"]}))
    packed = sorted(pathlib.Path("/kaggle/input").rglob("tsw.tar")) if KAGGLE else []
    if packed:  # an earlier session's work (learn: always; teach: to carry on where a stopped one left off)
        run("tar", "xf", packed[0], "-C", BASE)
    assert packed or MODE == "teach", "learn: add the teach notebook's output as input (tiny-voice-teach)"
    (WORKD / "deadline.json").unlink(missing_ok=True)  # a new session: a new 11-hour deadline
    run("bash", GPU / "setup.sh", env=env)
    try:
        run("bash", GPU / "run.sh", MODE, env=env)
    finally:  # kept even when a stage fails (20 Nov: three hours of the teacher's clips were lost to one error): a new session resumes
        if MODE == "teach":
            run("tar", "cf", OUT / "tsw.tar", "-C", BASE, "--exclude=tsw/venv", "tsw")
        else:
            if (WORKD / "result.tar.gz").exists():
                shutil.copy(WORKD / "result.tar.gz", OUT / "result.tar.gz")
            if (WORKD / "runs").exists():
                run("tar", "cf", OUT / "tsw.tar", "-C", BASE, "--exclude=tsw/venv", "tsw")  # to resume training, if it stopped early
else:
    raise SystemExit(f"MODE {MODE!r}?")


In [ ]:
# what came out
summary = OUT / "design/summary.json"
if summary.exists():
    s = json.loads(summary.read_text())
    print({k: v for k, v in s.items() if k != "candidates"})
    for c in s["candidates"]:
        print(c["id"], c["label"], "words wrong es", c.get("wer_es"), "en", c.get("wer_en"))
for p in sorted(OUT.rglob("*")):
    if p.is_file() and p.suffix in (".tar", ".gz", ".json"):
        print(f"{p.stat().st_size / 1e6:9.1f} MB  {p.relative_to(OUT)}")
for p in sorted((BASE / "tsw").glob("*/summary.json")) if (BASE / "tsw").exists() else []:
    print(p.parent.name, p.read_text()[:400])
